# Human-in-the-Loop, Uncertainty-Aware, Explanation-Stable Tabular ML Framework for NF1 Classification


**Aim:** Develop and validate a human-in-the-loop, uncertainty-aware, explanation-stable tabular ML framework for familial/sporadic NF1 classification under rare-disease small-sample conditions.

**Dataset Link:** https://archive.ics.uci.edu/dataset/1162/neurofibromatosis+type+1+clinical+symptoms+of+familial+and+sporadic+cases

| Component | Detail |
|-----------|--------|
| Dataset | UCI NF1 (n=296, 19 features) |
| Target | Case Type: 0=Sporadic, 1=Familial |
| Models | LR, SVM, RF, XGBoost, LightGBM, CatBoost, TabPFN |
| Framework | TabPFN + Conformal Prediction + SHAP Stability + HiTL Triage |
| Validation | Repeated Stratified CV, Bootstrap CI, Wilcoxon, McNemar |

---

## Section 0 — Install Dependencies

In [ ]:
import subprocess, sys

pkgs = [
    'tabpfn',
    'shap',
    'lime',
    'mapie',
    'lightgbm',
    'catboost',
    'xgboost',
    'imbalanced-learn',
    'scikit-learn',
    'scipy',
    'matplotlib',
    'seaborn',
    'pandas',
    'numpy',
    'openpyxl',
    'statsmodels',
    'mlxtend',
    'tqdm',
    'optuna',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=False)
print('Installation complete')

## Section 1 — Imports & Global Configuration

In [ ]:
import warnings, os, json, time, copy
warnings.filterwarnings('ignore')
os.makedirs('outputs/figures', exist_ok=True)
os.makedirs('outputs/tables', exist_ok=True)

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    StratifiedKFold, RepeatedStratifiedKFold, train_test_split)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef, brier_score_loss,
    roc_curve, precision_recall_curve, confusion_matrix)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

try:
    from tabpfn import TabPFNClassifier
    TABPFN_AVAILABLE = True
    print('TabPFN available')
except ImportError:
    TABPFN_AVAILABLE = False
    print('TabPFN not available - will use best baseline as framework model')

import shap
from lime.lime_tabular import LimeTabularExplainer
from scipy.stats import wilcoxon, mannwhitneyu, spearmanr, chi2_contingency

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
PALETTE = ['#2196F3', '#F44336', '#4CAF50', '#FF9800', '#9C27B0', '#00BCD4', '#FF5722']
print('Imports complete')

## Section 2 — Data Loading & Exploratory Data Analysis

In [ ]:
# Upload dataset-uci.xlsx to Colab via Files panel, or adjust path:
# from google.colab import files; files.upload()

DATA_PATH = 'dataset-uci.xlsx'   # adjust if needed

df_raw = pd.read_excel(DATA_PATH)
df_raw = df_raw.drop(columns=['Unnamed: 0'], errors='ignore')

TARGET = 'Case Type'
print(f'Dataset shape: {df_raw.shape}')
print(f'\nTarget distribution:\n{df_raw[TARGET].value_counts()}')
print(f'\nClass balance (prop. familial): {df_raw[TARGET].mean():.3f}')
print(f'\nMissing values:\n{df_raw.isnull().sum()[df_raw.isnull().sum()>0]}')
df_raw.describe()

In [ ]:
# Feature staging (leakage-aware design)
DEMOGRAPHIC_FEATURES = ['Age of Mother', 'Age of Father', 'Age at First Diagnosis']
CLINICAL_FEATURES = [
    'Tumour Case', 'Cafe au lait (CLS)' if 'Cafe au lait (CLS)' in df_raw.columns
    else 'Café au lait (CLS)',
    'Axillary Freckles', 'Inguinal Freckles', 'Lisch Nodules',
    'Dermal Neurofibromins', 'Plexiform Neurofibromins', 'Optic Glioma',
    'Skeletal Dysplasia', 'Learning Disability', 'Hypertension',
    'Astrocytoma', 'Hamartoma', 'Scoliosis', 'Other Symptoms'
]
# Auto-detect CLS column name
cls_col = [c for c in df_raw.columns if 'lait' in c.lower()]
if cls_col:
    CLINICAL_FEATURES = [cls_col[0] if 'lait' in f.lower() else f
                         for f in CLINICAL_FEATURES]

ALL_FEATURES = DEMOGRAPHIC_FEATURES + CLINICAL_FEATURES
# Validate features exist
ALL_FEATURES = [f for f in ALL_FEATURES if f in df_raw.columns]
DEMOGRAPHIC_FEATURES = [f for f in DEMOGRAPHIC_FEATURES if f in df_raw.columns]
CLINICAL_FEATURES = [f for f in CLINICAL_FEATURES if f in df_raw.columns]

FEATURE_SETS = {
    'All features':     ALL_FEATURES,
    'Clinical only':    CLINICAL_FEATURES,
    'Demographic only': DEMOGRAPHIC_FEATURES,
}
print('Feature stages:')
for k, v in FEATURE_SETS.items():
    print(f'  {k}: {len(v)} features -> {v}')

In [ ]:
# EDA: Missing values + class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5),dpi=600)

missing_pct = df_raw[ALL_FEATURES].isnull().mean() * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)
if len(missing_pct):
    axes[0].barh(missing_pct.index, missing_pct.values, color=PALETTE[0], alpha=0.8)
    for i, v in enumerate(missing_pct.values):
        axes[0].text(v + 0.2, i, f'{v:.1f}%', va='center', fontsize=9)
axes[0].set_xlabel('Missing (%)')
axes[0].set_title('Missing Values by Feature')

counts = df_raw[TARGET].value_counts().sort_index()
axes[1].bar(['Sporadic (0)', 'Familial (1)'], counts.values,
            color=[PALETTE[0], PALETTE[1]], alpha=0.85)
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 1, str(v), ha='center', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].set_title('Class Distribution')

plt.tight_layout()
plt.savefig('outputs/figures/fig1_eda.png', bbox_inches='tight')
plt.show()

In [ ]:
# EDA: Clinical feature prevalence by class
class_means = df_raw.groupby(TARGET)[CLINICAL_FEATURES].mean()
x = np.arange(len(CLINICAL_FEATURES))
fig, ax = plt.subplots(figsize=(14, 5),dpi=600)
ax.bar(x - 0.18, class_means.loc[0], 0.35, label='Sporadic (0)',
       color=PALETTE[0], alpha=0.8)
ax.bar(x + 0.18, class_means.loc[1], 0.35, label='Familial (1)',
       color=PALETTE[1], alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f.replace(' ', '\n') for f in CLINICAL_FEATURES], fontsize=7)
ax.set_ylabel('Proportion positive')
ax.set_title('Clinical Feature Prevalence by NF1 Class')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/figures/fig2_prevalence.png', bbox_inches='tight')
plt.show()

In [ ]:
# Spearman correlation heatmap
fig, ax = plt.subplots(figsize=(13, 10),dpi=600)
corr = df_raw[ALL_FEATURES + [TARGET]].corr(method='spearman')
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 7})
ax.set_title('Spearman Correlation Matrix')
plt.tight_layout()
plt.savefig('outputs/figures/fig3_correlation.png', bbox_inches='tight')
plt.show()

In [ ]:
# Table 1: Statistical group comparison
group0 = df_raw[df_raw[TARGET] == 0]
group1 = df_raw[df_raw[TARGET] == 1]
stat_rows = []
for feat in ALL_FEATURES:
    a = group0[feat].dropna()
    b = group1[feat].dropna()
    if df_raw[feat].nunique() <= 3:
        try:
            ct = pd.crosstab(df_raw[feat], df_raw[TARGET])
            _, p, _, _ = chi2_contingency(ct)
            test = 'Chi-square'
        except Exception:
            p, test = np.nan, 'N/A'
    else:
        _, p = mannwhitneyu(a, b, alternative='two-sided')
        test = 'Mann-Whitney U'
    stat_rows.append({
        'Feature': feat, 'Test': test,
        'Sporadic mean': round(a.mean(), 3),
        'Familial mean': round(b.mean(), 3),
        'p-value': round(p, 4),
        'Sig.': '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    })
stat_df = pd.DataFrame(stat_rows).sort_values('p-value')
stat_df.to_csv('outputs/tables/table1_group_comparison.csv', index=False)
print('Table 1 — Group Comparison:')
display(stat_df)

## Section 3 — Preprocessing & Model Definitions

In [ ]:
def make_preprocessor(feature_list):
    """KNN impute continuous, mode impute binary, scale continuous."""
    cont = [f for f in feature_list if f in DEMOGRAPHIC_FEATURES]
    binary = [f for f in feature_list if f in CLINICAL_FEATURES]
    transformers = []
    if cont:
        transformers.append(('cont', Pipeline([
            ('imp', KNNImputer(n_neighbors=5)),
            ('scl', StandardScaler())
        ]), cont))
    if binary:
        transformers.append(('bin', SimpleImputer(strategy='most_frequent'), binary))
    ct = ColumnTransformer(transformers, remainder='drop')
    return ct, cont + binary


def get_models():
    models = {
        'Logistic Regression': LogisticRegression(
            C=1.0, solver='lbfgs', max_iter=1000,
            class_weight='balanced', random_state=SEED),
        'SVM': CalibratedClassifierCV(
            SVC(kernel='rbf', C=1.0, gamma='scale',
                class_weight='balanced', random_state=SEED), cv=5),
        'Random Forest': RandomForestClassifier(
            n_estimators=500, class_weight='balanced',
            min_samples_leaf=2, random_state=SEED, n_jobs=-1),
        'XGBoost': XGBClassifier(
            n_estimators=200, learning_rate=0.05, max_depth=4,
            eval_metric='logloss', random_state=SEED, verbosity=0),
        'LightGBM': LGBMClassifier(
            n_estimators=200, learning_rate=0.05,
            class_weight='balanced', random_state=SEED, verbose=-1),
        'CatBoost': CatBoostClassifier(
            iterations=200, learning_rate=0.05, depth=4,
            auto_class_weights='Balanced', random_seed=SEED, verbose=0),
    }
    if TABPFN_AVAILABLE:
        models['TabPFN'] = TabPFNClassifier(device='cpu', N_ensemble_configurations=32)
    return models


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'ROC-AUC':      roc_auc_score(y_true, y_prob),
        'PR-AUC':       average_precision_score(y_true, y_prob),
        'Balanced Acc': balanced_accuracy_score(y_true, y_pred),
        'F1':           f1_score(y_true, y_pred, zero_division=0),
        'MCC':          matthews_corrcoef(y_true, y_pred),
        'Brier Score':  brier_score_loss(y_true, y_prob),
    }


def bootstrap_ci(scores, n_boot=1000, alpha=0.05):
    boot = [np.mean(np.random.choice(scores, len(scores), replace=True))
            for _ in range(n_boot)]
    return np.mean(scores), np.percentile(boot, 100*alpha/2), np.percentile(boot, 100*(1-alpha/2))


print('Preprocessor and models defined')
print('Models:', list(get_models().keys()))

## Section 4 — Repeated Stratified Cross-Validation

In [ ]:
N_SPLITS, N_REPEATS, N_BOOT = 5, 10, 1000
METRICS = ['ROC-AUC', 'PR-AUC', 'Balanced Acc', 'F1', 'MCC', 'Brier Score']


def run_rscv(df, feature_list, fs_name='All', n_splits=5, n_repeats=10):
    """Repeated stratified CV; returns per-fold scores + OOF probs."""
    X = df[feature_list].copy()
    y = df[TARGET].values
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats,
                                   random_state=SEED)
    models = get_models()
    results  = {m: {k: [] for k in METRICS} for m in models}
    oof_prob = {m: np.zeros(len(y)) for m in models}
    fold_cnt = {m: np.zeros(len(y), dtype=int) for m in models}

    total = n_splits * n_repeats
    print(f'Running {total} folds x {len(models)} models | Feature set: {fs_name}')
    for train_idx, test_idx in tqdm(rskf.split(X, y), total=total):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]
        pre, _ = make_preprocessor(feature_list)
        X_tr_pp = pre.fit_transform(X_tr)
        X_te_pp = pre.transform(X_te)
        for m_name, model in models.items():
            m = copy.deepcopy(model)
            try:
                m.fit(X_tr_pp, y_tr)
                proba = m.predict_proba(X_te_pp)[:, 1]
                for k, v in compute_metrics(y_te, proba).items():
                    results[m_name][k].append(v)
                oof_prob[m_name][test_idx] += proba
                fold_cnt[m_name][test_idx] += 1
            except Exception as e:
                print(f'  WARNING {m_name}: {e}')

    for m_name in models:
        cnt = np.maximum(fold_cnt[m_name], 1)
        oof_prob[m_name] /= cnt

    return results, oof_prob, y


# Run main experiment
cv_results, oof_probs, y_full = run_rscv(df_raw, ALL_FEATURES, 'All features')
print('Cross-validation complete')

In [ ]:
# Table 2: Main results with bootstrap 95% CI
summary_rows = []
for m_name, m_res in cv_results.items():
    row = {'Model': m_name}
    for metric in METRICS:
        if not m_res[metric]:
            row[metric] = 'N/A'; row[f'{metric}_m'] = -1
            continue
        mean, lo, hi = bootstrap_ci(m_res[metric])
        row[metric] = f'{mean:.3f} [{lo:.3f}–{hi:.3f}]'
        row[f'{metric}_m'] = mean
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values('ROC-AUC_m', ascending=False)
summary_df[['Model'] + METRICS].to_csv('outputs/tables/table2_main_results.csv', index=False)
mean_aucs = {m: r['ROC-AUC_m'] for m in summary_df['Model'].values
             for r in [summary_df[summary_df['Model']==m].to_dict('records')[0]]}
BEST_MODEL = max(mean_aucs, key=mean_aucs.get)
print(f'Best model: {BEST_MODEL} (ROC-AUC={mean_aucs[BEST_MODEL]:.3f})')
print('\nTable 2:')
display(summary_df[['Model'] + METRICS])

In [ ]:
# Figure: ROC & PR curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5),dpi=600)
for idx, (m_name, probs) in enumerate(oof_probs.items()):
    try:
        color = PALETTE[idx % len(PALETTE)]
        fpr, tpr, _ = roc_curve(y_full, probs)
        axes[0].plot(fpr, tpr, lw=1.8, color=color,
                     label=f'{m_name} ({roc_auc_score(y_full,probs):.3f})')
        prec, rec, _ = precision_recall_curve(y_full, probs)
        axes[1].plot(rec, prec, lw=1.8, color=color,
                     label=f'{m_name} (AP={average_precision_score(y_full,probs):.3f})')
    except Exception: pass

axes[0].plot([0,1],[0,1],'k--',lw=1,alpha=0.4)
axes[0].set(xlabel='FPR', ylabel='TPR', title='ROC Curves (OOF)')
axes[0].legend(fontsize=8, loc='lower right')
axes[1].axhline(y_full.mean(), color='k', ls='--', lw=1, alpha=0.4)
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curves (OOF)')
axes[1].legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.savefig('outputs/figures/fig4_roc_pr.png', bbox_inches='tight')
plt.show()

In [ ]:
# Figure: Performance boxplots
fig, axes = plt.subplots(2, 3, figsize=(16, 9),dpi=600)
for i, metric in enumerate(METRICS):
    ax = axes.flatten()[i]
    data = [cv_results[m][metric] for m in cv_results if cv_results[m][metric]]
    labels = [m for m in cv_results if cv_results[m][metric]]
    bp = ax.boxplot(data, patch_artist=True, notch=True,
                    medianprops={'color':'black','linewidth':2})
    for patch, color in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=7)
    ax.set_title(metric)
    if metric == 'Brier Score': ax.invert_yaxis()
plt.suptitle('Model Performance Distribution (Repeated Stratified CV)', y=1.01)
plt.tight_layout()
plt.savefig('outputs/figures/fig5_boxplots.png', bbox_inches='tight')
plt.show()

## Section 5 — Statistical Validation

In [ ]:
# Wilcoxon signed-rank test: best model vs all others
REFERENCE = 'TabPFN' if TABPFN_AVAILABLE else BEST_MODEL
ref_scores = cv_results[REFERENCE]['ROC-AUC']

wilcoxon_rows = []
for m_name, m_res in cv_results.items():
    if m_name == REFERENCE or not m_res['ROC-AUC']: continue
    comp = m_res['ROC-AUC']
    if len(ref_scores) != len(comp): continue
    try:
        stat, p = wilcoxon(ref_scores, comp, alternative='greater')
    except Exception:
        stat, p = np.nan, np.nan
    wilcoxon_rows.append({
        'Comparison': f'{REFERENCE} vs {m_name}',
        'W': round(stat, 2) if not np.isnan(stat) else 'N/A',
        'p-value': round(p, 4) if not np.isnan(p) else 'N/A',
        'Significant (α=0.05)': 'Yes' if (not np.isnan(p) and p < 0.05) else 'No'
    })

wilcoxon_df = pd.DataFrame(wilcoxon_rows)
wilcoxon_df.to_csv('outputs/tables/table3_wilcoxon.csv', index=False)
print(f'Table 3 — Wilcoxon Tests ({REFERENCE} vs others, ROC-AUC):')
display(wilcoxon_df)

In [ ]:
# McNemar test: top-2 models
top2 = sorted(mean_aucs, key=mean_aucs.get, reverse=True)[:2]
print(f'McNemar: {top2[0]} vs {top2[1]}')
p1 = (oof_probs[top2[0]] >= 0.5).astype(int)
p2 = (oof_probs[top2[1]] >= 0.5).astype(int)
c1 = (p1 == y_full); c2 = (p2 == y_full)
n10 = (c1 & ~c2).sum(); n01 = (~c1 & c2).sum()
print(f'Contingency: n10={n10}, n01={n01}')
if n10 + n01 > 0:
    chi2_mc = (abs(n10 - n01) - 1)**2 / (n10 + n01)
    p_mc = 1 - stats.chi2.cdf(chi2_mc, df=1)
    print(f'McNemar chi2={chi2_mc:.3f}, p={p_mc:.4f}')
    print('Significant:', 'Yes' if p_mc < 0.05 else 'No')
else:
    print('Identical predictions — McNemar not applicable')

## Section 6 — Calibration Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7),dpi=600)
ax.plot([0,1],[0,1],'k--',lw=1.5,label='Perfect calibration')
for idx, (m_name, probs) in enumerate(oof_probs.items()):
    try:
        frac_pos, mean_pred = calibration_curve(y_full, probs, n_bins=8)
        brier = brier_score_loss(y_full, probs)
        ax.plot(mean_pred, frac_pos, 'o-', lw=2, markersize=5,
                color=PALETTE[idx%len(PALETTE)],
                label=f'{m_name} (Brier={brier:.3f})')
    except Exception: pass
ax.set(xlabel='Mean Predicted Prob.', ylabel='Fraction of Positives',
       title='Calibration Curves (Reliability Diagram)')
ax.legend(fontsize=9); ax.set_xlim([0,1]); ax.set_ylim([0,1])
plt.tight_layout()
plt.savefig('outputs/figures/fig6_calibration.png', bbox_inches='tight')
plt.show()

## Section 7 — Inductive Conformal Prediction (Uncertainty Quantification)

In [ ]:
CONFORMAL_ALPHA = 0.10   # 90% coverage guarantee

def inductive_conformal_prediction(model_name, df, feature_list,
                                   alpha=CONFORMAL_ALPHA, n_trials=50):
    """
    Repeated split ICP.
    Coverage guarantee: P(Y in C(X)) >= 1 - alpha (marginal validity).
    Returns empirical coverage, avg set size, referral rate.
    """
    X = df[feature_list].copy()
    y = df[TARGET].values
    models = get_models()
    coverages, set_sizes, ref_rates = [], [], []

    for trial in range(n_trials):
        X_tmp, X_te, y_tmp, y_te = train_test_split(
            X, y, test_size=0.20, stratify=y, random_state=trial)
        X_tr, X_cal, y_tr, y_cal = train_test_split(
            X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=trial)

        pre, _ = make_preprocessor(feature_list)
        X_tr_pp  = pre.fit_transform(X_tr)
        X_cal_pp = pre.transform(X_cal)
        X_te_pp  = pre.transform(X_te)

        m = copy.deepcopy(models[model_name])
        try: m.fit(X_tr_pp, y_tr)
        except Exception: continue

        # Nonconformity: 1 - P(true class)
        nc_cal = 1 - m.predict_proba(X_cal_pp)[np.arange(len(y_cal)), y_cal]
        q = min(np.ceil((1-alpha)*(len(y_cal)+1))/len(y_cal), 1.0)
        tau = np.quantile(nc_cal, q)

        # Prediction sets on test
        pp_te = m.predict_proba(X_te_pp)
        in0 = (1 - pp_te[:,0]) <= tau
        in1 = (1 - pp_te[:,1]) <= tau
        pred_sets = [([0] if in0[i] else []) + ([1] if in1[i] else [])
                     for i in range(len(y_te))]

        sizes = [len(ps) for ps in pred_sets]
        coverages.append(np.mean([y_te[i] in pred_sets[i] for i in range(len(y_te))]))
        set_sizes.append(np.mean(sizes))
        ref_rates.append(np.mean([s != 1 for s in sizes]))

    return {
        'Coverage':     (np.mean(coverages), np.std(coverages)),
        'Set Size':     (np.mean(set_sizes),  np.std(set_sizes)),
        'Referral Rate':(np.mean(ref_rates),  np.std(ref_rates)),
    }

print('Running conformal prediction (50 trials per model)...')
conf_results = {}
for m_name in tqdm(list(cv_results.keys())):
    if cv_results[m_name]['ROC-AUC']:
        conf_results[m_name] = inductive_conformal_prediction(m_name, df_raw, ALL_FEATURES)
print('Conformal analysis complete')

In [ ]:
# Table 4: Conformal results
conf_rows = []
for m_name, res in conf_results.items():
    conf_rows.append({
        'Model': m_name,
        'Coverage (mean±std)': f"{res['Coverage'][0]:.3f} ± {res['Coverage'][1]:.3f}",
        'Avg Set Size':        f"{res['Set Size'][0]:.3f} ± {res['Set Size'][1]:.3f}",
        'Referral Rate':       f"{res['Referral Rate'][0]:.3f} ± {res['Referral Rate'][1]:.3f}",
        f'Target (≥{1-CONFORMAL_ALPHA:.0%})': 'Met' if res['Coverage'][0] >= 1-CONFORMAL_ALPHA else 'Not met',
    })
conf_df = pd.DataFrame(conf_rows)
conf_df.to_csv('outputs/tables/table4_conformal.csv', index=False)
print(f'Table 4 — Conformal Prediction (alpha={CONFORMAL_ALPHA}):')
display(conf_df)

In [ ]:
# Figure: Coverage vs referral rate
fig, ax = plt.subplots(figsize=(8, 6),dpi=600)
for idx, (m_name, res) in enumerate(conf_results.items()):
    ax.scatter(res['Referral Rate'][0], res['Coverage'][0],
               s=150, color=PALETTE[idx%len(PALETTE)],
               zorder=3, edgecolors='black', lw=0.8)
    ax.annotate(m_name, (res['Referral Rate'][0], res['Coverage'][0]),
                textcoords='offset points', xytext=(5,5), fontsize=9)
ax.axhline(1-CONFORMAL_ALPHA, color='red', ls='--', lw=1.5,
           label=f'{1-CONFORMAL_ALPHA:.0%} target coverage')
ax.set(xlabel='Referral Rate', ylabel='Empirical Coverage',
       title='Conformal Prediction: Coverage vs Referral Rate')
ax.legend(); ax.set_xlim([-0.05,1.05]); ax.set_ylim([0.5,1.05])
plt.tight_layout()
plt.savefig('outputs/figures/fig7_conformal.png', bbox_inches='tight')
plt.show()

## Section 8 — Human-in-the-Loop (HiTL) Triage Framework

In [ ]:
LOW_THRESH  = 0.25
HIGH_THRESH = 0.75

def hitl_triage(y_true, y_prob, low=LOW_THRESH, high=HIGH_THRESH):
    conf_mask = (y_prob < low) | (y_prob > high)
    uncer_mask = ~conf_mask
    y_pred_conf = (y_prob[conf_mask] >= 0.5).astype(int)
    conf_acc = (y_pred_conf == y_true[conf_mask]).mean() if conf_mask.sum() > 0 else np.nan
    return {
        'Referral Rate':      uncer_mask.mean(),
        'Confident Count':    int(conf_mask.sum()),
        'Uncertain Count':    int(uncer_mask.sum()),
        'Confident Accuracy': conf_acc,
        'Overall Accuracy':   ((y_prob >= 0.5).astype(int) == y_true).mean(),
    }

hitl_rows = []
for m_name, probs in oof_probs.items():
    if not cv_results[m_name]['ROC-AUC']: continue
    hitl_rows.append({'Model': m_name, **hitl_triage(y_full, probs)})

hitl_df = pd.DataFrame(hitl_rows)
hitl_df.to_csv('outputs/tables/table5_hitl.csv', index=False)
print(f'Table 5 — HiTL Triage (thresholds: [{LOW_THRESH},{HIGH_THRESH}]):')
display(hitl_df.round(4))

In [ ]:
# Threshold sweep: accuracy-referral tradeoff
probs_best = oof_probs[BEST_MODEL]
thresholds = np.arange(0.05, 0.51, 0.05)
td_rows = [{'t': t, **hitl_triage(y_full, probs_best, low=t, high=1-t)}
           for t in thresholds]
td_df = pd.DataFrame(td_rows).dropna(subset=['Confident Accuracy'])

fig, ax = plt.subplots(figsize=(9, 5),dpi=600)
ax2 = ax.twinx()
ax.plot(td_df['Referral Rate'], td_df['Confident Accuracy'],
        'b-o', lw=2.5, label='Confident Accuracy (left)')
ax2.plot(td_df['Referral Rate'], td_df['Confident Count'],
         'r--s', lw=2, label='# Confident cases (right)')
ax.set(xlabel='Referral Rate', ylabel='Confident Accuracy',
       title=f'HiTL Trade-off — {BEST_MODEL}')
ax2.set_ylabel('# Confident Predictions', color='r')
lines = ax.get_lines() + ax2.get_lines()
ax.legend(lines, [l.get_label() for l in lines], loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('outputs/figures/fig8_hitl_tradeoff.png', bbox_inches='tight')
plt.show()

## Section 9 — SHAP Explainability

In [ ]:
# Fit tree models on full dataset for XAI
X_all = df_raw[ALL_FEATURES].copy()
y_all = df_raw[TARGET].values
pre_final, ordered_feats = make_preprocessor(ALL_FEATURES)
X_all_pp = pre_final.fit_transform(X_all)
feature_names_pp = ordered_feats

xai_models = {}
for m_name in ['Random Forest', 'XGBoost', 'LightGBM', 'CatBoost']:
    m = copy.deepcopy(get_models()[m_name])
    m.fit(X_all_pp, y_all)
    xai_models[m_name] = m

# Pick best tree model
tree_models_ranked = sorted(
    [m for m in ['XGBoost','LightGBM','CatBoost','Random Forest'] if m in mean_aucs],
    key=lambda m: mean_aucs.get(m, 0), reverse=True)
BEST_TREE = tree_models_ranked[0]
print(f'Computing SHAP for: {BEST_TREE}')

explainer = shap.TreeExplainer(xai_models[BEST_TREE])
shap_values = explainer.shap_values(X_all_pp)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values
shap_df = pd.DataFrame(sv, columns=feature_names_pp)
X_all_pp_df = pd.DataFrame(X_all_pp, columns=feature_names_pp)
mean_abs_shap = np.abs(shap_df).mean().sort_values(ascending=True)
print('SHAP computed. Shape:', shap_df.shape)

In [ ]:
# SHAP bar + beeswarm
fig, axes = plt.subplots(1, 2, figsize=(16, 7),dpi=600)
axes[0].barh(mean_abs_shap.index, mean_abs_shap.values,
             color=PALETTE[0], alpha=0.8)
axes[0].set(xlabel='Mean |SHAP|', title=f'Global Feature Importance ({BEST_TREE})')

top10 = mean_abs_shap.tail(10).index.tolist()
for fi, feat in enumerate(top10):
    feat_vals = X_all_pp_df[feat].values
    shap_vals = shap_df[feat].values
    y_jitter  = fi + np.random.uniform(-0.3, 0.3, len(shap_vals))
    sc = axes[1].scatter(shap_vals, y_jitter, c=feat_vals,
                         cmap='coolwarm', s=15, alpha=0.7)
axes[1].set_yticks(range(len(top10)))
axes[1].set_yticklabels(top10, fontsize=9)
axes[1].set(xlabel='SHAP value', title='SHAP Beeswarm (Top 10)')
axes[1].axvline(0, color='k', lw=0.8, ls='--')
plt.colorbar(sc, ax=axes[1], label='Feature value (normalized)')
plt.suptitle(f'SHAP Analysis — {BEST_TREE}', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('outputs/figures/fig9_shap.png', bbox_inches='tight')
plt.show()

## Section 10 — Explanation Stability Analysis

In [ ]:
STABILITY_K = 5

def compute_explanation_stability(model_name, df, feature_list, n_folds=10):
    """
    Pairwise Spearman rank correlation + Jaccard@k of top-k features
    across stratified CV folds — measures explanation robustness.
    """
    X = df[feature_list].copy()
    y = df[TARGET].values
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    models = get_models()
    importance_list = []
    for train_idx, _ in tqdm(skf.split(X, y), total=n_folds,
                              desc=f'Stability [{model_name}]'):
        X_tr = X.iloc[train_idx]; y_tr = y[train_idx]
        pre, _ = make_preprocessor(feature_list)
        X_tr_pp = pre.fit_transform(X_tr)
        m = copy.deepcopy(models[model_name])
        try:
            m.fit(X_tr_pp, y_tr)
            exp = shap.TreeExplainer(m)
            sv  = exp.shap_values(X_tr_pp)
            if isinstance(sv, list): sv = sv[1]
            importance_list.append(np.abs(sv).mean(axis=0))
        except Exception: pass

    if len(importance_list) < 2:
        return {'Spearman (mean)': np.nan, 'Spearman (std)': np.nan,
                f'Jaccard@{STABILITY_K} (mean)': np.nan,
                f'Jaccard@{STABILITY_K} (std)': np.nan}

    spearmans, jaccards = [], []
    for i in range(len(importance_list)):
        for j in range(i+1, len(importance_list)):
            r, _ = spearmanr(importance_list[i], importance_list[j])
            spearmans.append(r)
            ti = set(np.argsort(importance_list[i])[-STABILITY_K:])
            tj = set(np.argsort(importance_list[j])[-STABILITY_K:])
            jaccards.append(len(ti & tj) / len(ti | tj))

    return {
        'Spearman (mean)':  round(np.mean(spearmans), 3),
        'Spearman (std)':   round(np.std(spearmans), 3),
        f'Jaccard@{STABILITY_K} (mean)': round(np.mean(jaccards), 3),
        f'Jaccard@{STABILITY_K} (std)':  round(np.std(jaccards), 3),
    }

stability_results = {}
for m_name in ['Random Forest', 'XGBoost', 'LightGBM', 'CatBoost']:
    if cv_results[m_name]['ROC-AUC']:
        stability_results[m_name] = compute_explanation_stability(
            m_name, df_raw, ALL_FEATURES)

stab_df = pd.DataFrame([{'Model': k, **v} for k, v in stability_results.items()])
stab_df.to_csv('outputs/tables/table6_stability.csv', index=False)
print('Table 6 — Explanation Stability:')
display(stab_df)

In [ ]:
# Figure: stability comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5),dpi=600)
x = np.arange(len(stab_df))
for ai, (col, label) in enumerate([
    ('Spearman (mean)', 'Spearman Rank Corr.'),
    (f'Jaccard@{STABILITY_K} (mean)', f'Jaccard@{STABILITY_K}')
]):
    ax = axes[ai]
    err_col = col.replace('mean', 'std')
    errs = stab_df[err_col].values if err_col in stab_df else np.zeros(len(stab_df))
    ax.bar(x, stab_df[col].values, color=PALETTE[:len(x)], alpha=0.8,
           yerr=errs, capsize=5)
    ax.set_xticks(x)
    ax.set_xticklabels(stab_df['Model'], rotation=25, ha='right')
    ax.set(ylabel=label, title=f'Explanation Stability: {label}', ylim=[0, 1.15])
    ax.axhline(1.0, color='gray', ls='--', lw=0.8, alpha=0.5)
plt.tight_layout()
plt.savefig('outputs/figures/fig10_stability.png', bbox_inches='tight')
plt.show()

## Section 11 — LIME Local Explanations

In [ ]:
lime_explainer = LimeTabularExplainer(
    X_all_pp,
    feature_names=feature_names_pp,
    class_names=['Sporadic', 'Familial'],
    mode='classification',
    random_state=SEED)

best_tree_model = xai_models[BEST_TREE]
proba_full = best_tree_model.predict_proba(X_all_pp)[:, 1]

# Select 3 representative cases
correct = proba_full.round() == y_all
conf    = np.abs(proba_full - 0.5) > 0.3

idx_cases = {
    'Confident Correct\n(Sporadic)':   int(np.where(correct & conf & (y_all==0))[0][0]),
    'Confident Correct\n(Familial)':   int(np.where(correct & conf & (y_all==1))[0][0]),
    'Uncertain (Refer)': int(np.argmin(np.abs(proba_full - 0.5))),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6),dpi=600)
for ax, (label, idx) in zip(axes, idx_cases.items()):
    exp = lime_explainer.explain_instance(
        X_all_pp[idx], best_tree_model.predict_proba, num_features=8)
    fv = exp.as_list()
    feats_ = [x[0] for x in fv]
    vals_  = [x[1] for x in fv]
    colors_ = [PALETTE[1] if v > 0 else PALETTE[0] for v in vals_]
    ax.barh(feats_, vals_, color=colors_, alpha=0.85)
    ax.axvline(0, color='k', lw=0.8)
    ax.set_title(
        f'{label}\nTrue: {["Sporadic","Familial"][y_all[idx]]}  '
        f'P(Fam)={proba_full[idx]:.2f}', fontsize=9)
    ax.set_xlabel('LIME contribution')
    ax.tick_params(labelsize=7)

plt.suptitle(f'LIME Local Explanations — {BEST_TREE}', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/figures/fig11_lime.png', bbox_inches='tight')
plt.show()

## Section 12 — Sensitivity Analysis Across Feature Sets

In [ ]:
sensitivity_results = {}
for fs_name, fs_feats in FEATURE_SETS.items():
    if len(fs_feats) < 2: continue
    print(f'\nFeature set: {fs_name}')
    cv_res, _, _ = run_rscv(df_raw, fs_feats, fs_name, n_splits=5, n_repeats=5)
    sensitivity_results[fs_name] = cv_res
print('\nSensitivity analysis complete')

In [ ]:
# Sensitivity heatmap
sens_rows = []
for fs_name, cv_res in sensitivity_results.items():
    row = {'Feature Set': fs_name}
    for m_name, m_res in cv_res.items():
        row[m_name] = round(np.mean(m_res['ROC-AUC']), 3) if m_res['ROC-AUC'] else np.nan
    sens_rows.append(row)

sens_df = pd.DataFrame(sens_rows).set_index('Feature Set')
sens_df.to_csv('outputs/tables/table7_sensitivity.csv')

fig, ax = plt.subplots(figsize=(11, 4),dpi=600)
sns.heatmap(sens_df.astype(float), annot=True, fmt='.3f',
            cmap='YlOrRd', vmin=0.5, vmax=1.0,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'ROC-AUC'})
ax.set_title('Sensitivity Analysis: ROC-AUC × Feature Set × Model')
plt.tight_layout()
plt.savefig('outputs/figures/fig12_sensitivity.png', bbox_inches='tight')
plt.show()

## Section 13 — Proposed Framework: TabPFN + Conformal + HiTL

In [ ]:
FRAMEWORK_MODEL = 'TabPFN' if TABPFN_AVAILABLE else BEST_MODEL
print(f'Framework model: {FRAMEWORK_MODEL}')

fw_res = {'aucs': [], 'conf_acc': [], 'ref_rate': [], 'coverage': [], 'set_size': []}
rskf_fw = RepeatedStratifiedKFold(n_splits=5, n_repeats=20, random_state=SEED)
X_fw = df_raw[ALL_FEATURES].copy()
y_fw = df_raw[TARGET].values

print('Running framework evaluation (100 folds)...')
for train_idx, test_idx in tqdm(rskf_fw.split(X_fw, y_fw), total=100):
    X_tr, X_te = X_fw.iloc[train_idx], X_fw.iloc[test_idx]
    y_tr, y_te = y_fw[train_idx], y_fw[test_idx]

    n_cal = max(int(len(y_tr) * 0.2), 5)
    cal_idx  = np.random.choice(len(y_tr), n_cal, replace=False)
    prop_idx = np.setdiff1d(np.arange(len(y_tr)), cal_idx)

    pre, _ = make_preprocessor(ALL_FEATURES)
    X_prop_pp = pre.fit_transform(X_tr.iloc[prop_idx])
    X_cal_pp  = pre.transform(X_tr.iloc[cal_idx])
    X_te_pp   = pre.transform(X_te)
    y_prop, y_cal = y_tr[prop_idx], y_tr[cal_idx]

    m = copy.deepcopy(get_models()[FRAMEWORK_MODEL])
    try:
        m.fit(X_prop_pp, y_prop)
    except Exception: continue

    # Stage 1: Predict
    proba_te = m.predict_proba(X_te_pp)[:, 1]
    if len(np.unique(y_te)) > 1:
        fw_res['aucs'].append(roc_auc_score(y_te, proba_te))

    # Stage 2: Conformal sets
    nc_cal = 1 - m.predict_proba(X_cal_pp)[np.arange(len(y_cal)), y_cal]
    q = min(np.ceil((1-CONFORMAL_ALPHA)*(len(y_cal)+1))/len(y_cal), 1.0)
    tau = np.quantile(nc_cal, q)
    pp2 = m.predict_proba(X_te_pp)
    in0 = (1 - pp2[:,0]) <= tau
    in1 = (1 - pp2[:,1]) <= tau
    pred_sets = [([0] if in0[i] else []) + ([1] if in1[i] else [])
                 for i in range(len(y_te))]
    sizes = [len(ps) for ps in pred_sets]
    fw_res['coverage'].append(np.mean([y_te[i] in pred_sets[i] for i in range(len(y_te))]))
    fw_res['set_size'].append(np.mean(sizes))

    # Stage 3: HiTL
    ref_mask = np.array([s != 1 for s in sizes])
    fw_res['ref_rate'].append(ref_mask.mean())
    if (~ref_mask).sum() > 0:
        fw_res['conf_acc'].append(
            (proba_te[~ref_mask].round() == y_te[~ref_mask]).mean())

print('Framework evaluation complete')

In [ ]:
# Table 8 + print
print('PROPOSED FRAMEWORK — FINAL RESULTS')
fw_summary = {}
for key, label in [
    ('aucs',     'ROC-AUC'),
    ('conf_acc', 'Confident-Case Accuracy'),
    ('ref_rate', 'Referral Rate (→ Clinician)'),
    ('coverage', 'Conformal Coverage'),
    ('set_size', 'Avg Prediction Set Size'),
]:
    vals = [v for v in fw_res[key] if not np.isnan(v)]
    if vals:
        mean, lo, hi = bootstrap_ci(vals)
        fw_summary[label] = f'{mean:.3f} [95% CI: {lo:.3f}–{hi:.3f}]'
        print(f'{label:40s}: {fw_summary[label]}')

pd.DataFrame([fw_summary]).to_csv('outputs/tables/table8_framework.csv', index=False)

In [ ]:
# HPO with Optuna (XGBoost ablation)
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

X_hpo = pre_final.transform(df_raw[ALL_FEATURES].copy())
y_hpo = df_raw[TARGET].values
cv_hpo = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def objective(trial):
    params = {
        'n_estimators':  trial.suggest_int('n_estimators', 100, 600, step=50),
        'max_depth':     trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample':     trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':     trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':    trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }
    model = XGBClassifier(**params, eval_metric='logloss',
                          random_state=SEED, verbosity=0)
    scores = [roc_auc_score(y_hpo[te], model.fit(X_hpo[tr], y_hpo[tr])
                            .predict_proba(X_hpo[te])[:,1])
              for tr, te in cv_hpo.split(X_hpo, y_hpo)]
    return np.mean(scores)

study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=50, show_progress_bar=True)
print(f'Best HPO ROC-AUC: {study.best_value:.4f}')
print('Best params:', study.best_params)

fig, ax = plt.subplots(figsize=(10, 4),dpi=600)
trial_vals = [t.value for t in study.trials]
ax.plot(trial_vals, 'o', alpha=0.4, markersize=4, color=PALETTE[0])
ax.plot(np.maximum.accumulate(trial_vals), '-', lw=2.5, color=PALETTE[1])
ax.set(xlabel='Trial #', ylabel='CV ROC-AUC', title='Optuna HPO History')
plt.tight_layout()
plt.savefig('outputs/figures/fig13_hpo.png', bbox_inches='tight')
plt.show()

## Section 14 — Consolidated Results & Main Figure Panel

In [ ]:
fig = plt.figure(figsize=(20, 16),dpi=600)
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.42, wspace=0.35)

# (A) Class distribution
ax_a = fig.add_subplot(gs[0, 0])
counts = df_raw[TARGET].value_counts().sort_index()
ax_a.bar(['Sporadic', 'Familial'], counts.values,
         color=[PALETTE[0], PALETTE[1]], alpha=0.85)
for i, v in enumerate(counts.values):
    ax_a.text(i, v+1, str(v), ha='center', fontweight='bold')
ax_a.set(title='(A) Class Distribution', ylabel='Count')

# (B) ROC curves
ax_b = fig.add_subplot(gs[0, 1:3])
for idx, (m_name, probs) in enumerate(oof_probs.items()):
    try:
        fpr, tpr, _ = roc_curve(y_full, probs)
        ax_b.plot(fpr, tpr, lw=1.8, color=PALETTE[idx%len(PALETTE)],
                  label=f'{m_name} ({roc_auc_score(y_full,probs):.3f})')
    except Exception: pass
ax_b.plot([0,1],[0,1],'k--',lw=1,alpha=0.4)
ax_b.set(title='(B) ROC Curves', xlabel='FPR', ylabel='TPR')
ax_b.legend(fontsize=7, loc='lower right')

# (C) Calibration
ax_c = fig.add_subplot(gs[1, 0])
ax_c.plot([0,1],[0,1],'k--',lw=1)
for idx, (m_name, probs) in enumerate(list(oof_probs.items())[:5]):
    try:
        fp, mp = calibration_curve(y_full, probs, n_bins=7)
        ax_c.plot(mp, fp, 'o-', lw=1.5, markersize=4,
                  color=PALETTE[idx%len(PALETTE)], label=m_name)
    except Exception: pass
ax_c.set(title='(C) Calibration', xlabel='Predicted', ylabel='Observed')
ax_c.legend(fontsize=7)

# (D) SHAP importance
ax_d = fig.add_subplot(gs[1, 1])
top10 = mean_abs_shap.tail(10)
ax_d.barh(top10.index, top10.values, color=PALETTE[2], alpha=0.8)
ax_d.set(title=f'(D) SHAP Importance\n({BEST_TREE})', xlabel='Mean |SHAP|')
ax_d.tick_params(labelsize=7)

# (E) Conformal
ax_e = fig.add_subplot(gs[1, 2])
for idx, (m_name, res) in enumerate(conf_results.items()):
    ax_e.scatter(res['Referral Rate'][0], res['Coverage'][0],
                 s=130, color=PALETTE[idx%len(PALETTE)],
                 zorder=3, edgecolors='k', lw=0.8)
    ax_e.annotate(m_name, (res['Referral Rate'][0], res['Coverage'][0]),
                  xytext=(3,3), textcoords='offset points', fontsize=7)
ax_e.axhline(1-CONFORMAL_ALPHA, color='r', ls='--', lw=1.5)
ax_e.set(title='(E) Conformal Coverage vs Referral',
         xlabel='Referral Rate', ylabel='Coverage')

# (F) Sensitivity heatmap
ax_f = fig.add_subplot(gs[2, 0:2])
sns.heatmap(sens_df.astype(float), annot=True, fmt='.3f',
            cmap='YlOrRd', vmin=0.5, vmax=1.0,
            linewidths=0.5, ax=ax_f, annot_kws={'size': 8})
ax_f.set_title('(F) Sensitivity: ROC-AUC × Feature Set')

# (G) HiTL tradeoff
ax_g = fig.add_subplot(gs[2, 2])
ax_g.plot(td_df['Referral Rate'], td_df['Confident Accuracy'], 'b-o', lw=2.5)
ax_g.set(title=f'(G) HiTL Accuracy-Referral Tradeoff\n({BEST_MODEL})',
         xlabel='Referral Rate', ylabel='Confident Accuracy')

plt.suptitle(
    'NF1 Classification Framework — Comprehensive Results\n'
    '(Familial vs Sporadic · UCI Dataset · n=296)',
    fontsize=14, y=1.01, fontweight='bold')
plt.savefig('outputs/figures/fig_main_panel.png', bbox_inches='tight', dpi=200)
plt.show()
print('Main panel saved')

## Section 15 — Reproducibility & Download

In [ ]:
import pkg_resources, platform

env_info = {
    'python': platform.python_version(),
    'random_seed': SEED,
    'n_splits': N_SPLITS,
    'n_repeats': N_REPEATS,
    'n_bootstrap': N_BOOT,
    'conformal_alpha': CONFORMAL_ALPHA,
    'hitl_thresholds': [LOW_THRESH, HIGH_THRESH],
    'packages': {p.project_name: p.version for p in pkg_resources.working_set
                 if p.project_name in [
                     'scikit-learn','xgboost','lightgbm','catboost',
                     'tabpfn','shap','lime','mapie','optuna',
                     'pandas','numpy','scipy','matplotlib','seaborn'
                 ]},
}
with open('outputs/reproducibility_info.json', 'w') as f:
    json.dump(env_info, f, indent=2)
print('\nEnvironment saved. Output files:')
for root, dirs, files in os.walk('outputs'):
    for fname in sorted(files):
        fpath = os.path.join(root, fname)
        print(f'  {fpath}  ({os.path.getsize(fpath)/1024:.1f} KB)')

In [ ]:
# Download all outputs as zip (Colab)
import shutil
shutil.make_archive('NF1_framework_outputs', 'zip', 'outputs')
try:
    from google.colab import files
    files.download('NF1_framework_outputs.zip')
    print('Downloading outputs...')
except Exception:
    print('Not in Colab - outputs/ folder is available locally.')